In [1]:
# !pip uninstall flash-attn
!pip install open_clip_torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
 

# import libary

In [2]:
import os
import pickle
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPImageProcessor
from transformers import CLIPModel, CLIPProcessor
import open_clip 

2025-10-11 18:11:16.578080: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760206276.775157      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760206276.834046      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `ty

## Paths

In [3]:
BASE_DIR = '/kaggle/input/data-nlp-bai-2/img'
OUTPUT_DIR = '/kaggle/working/'
IMAGE_DIR = os.path.join(BASE_DIR, 'img')          # kiểm tra lại layout: .../img/img nếu cần
FEATURES_FILE = os.path.join(OUTPUT_DIR, 'clip_features.pkl')
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Dataset

In [5]:
class ImageDataset(Dataset):
    def __init__(self, image_dir):
        self.image_dir = image_dir
        self.image_files = [
            f for f in os.listdir(image_dir)
            if f.lower().endswith(('.jpg','.jpeg','.png')) and f not in ['6801.jpg','6803.jpg']
        ]
    def __len__(self): return len(self.image_files)
    def __getitem__(self, idx):
        fn = self.image_files[idx]
        img = Image.open(os.path.join(self.image_dir, fn)).convert("RGB")
        return img, os.path.splitext(fn)[0]

def collate_pil(batch):
    imgs, ids = zip(*batch)
    return list(imgs), list(ids)

## OpenCLIP

In [6]:
model_name = 'ViT-B-32'
pretrained  = 'laion2b_s34b_b79k'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained, device=device)
model.eval()

# ===== Hàm trích batch =====
@torch.no_grad()
def extract_batch_openclip(pil_images, device):
    # preprocess trả sẵn tensor [B,3,224,224] (tuỳ backbone); stack thủ công
    batch = torch.stack([preprocess(img) for img in pil_images], dim=0).to(device)
    # AMP để tiết kiệm VRAM
    use_amp = torch.cuda.is_available()
    with torch.cuda.amp.autocast(enabled=use_amp):
        feats = model.encode_image(batch)          # [B, D]
    feats = feats / feats.norm(dim=-1, keepdim=True)
    return feats.detach().cpu().numpy()            # (B, D)


open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

## Loop DataLoader

In [7]:
loader = DataLoader(ImageDataset(IMAGE_DIR),
                    batch_size=16, shuffle=False, num_workers=2,
                    pin_memory=True, collate_fn=collate_pil)

features = {}
for pil_batch, id_batch in tqdm(loader, desc="Extracting CLIP (open_clip)"):
    feats = extract_batch_openclip(pil_batch, device)    # (B, D)
    for k, v in zip(id_batch, feats):
        features[k] = v[np.newaxis, :]  
        
with open(FEATURES_FILE, 'wb') as f:
    pickle.dump(features, f)

print("Saved:", FEATURES_FILE)

Extracting CLIP (open_clip):   0%|          | 0/3241 [00:00<?, ?it/s]/tmp/ipykernel_19/3891794650.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
Extracting CLIP (open_clip): 100%|██████████| 3241/3241 [11:50<00:00,  4.56it/s]


Saved: /kaggle/working/clip_features.pkl


In [8]:
# import os
# import pickle
# import numpy as np
# from PIL import Image
# from tqdm import tqdm
# import torch
# from torch.utils.data import Dataset, DataLoader
# import open_clip  # <- OpenCLIP

# # ===== Paths =====
# BASE_DIR = '/kaggle/input/data-nlp-bai-2/img'
# IMAGE_DIR = os.path.join(BASE_DIR, 'img')   # sửa nếu layout khác
# OUTPUT_DIR = '/kaggle/working'
# FEATURES_FILE = os.path.join(OUTPUT_DIR, 'clip_features.pkl')
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # ===== Dataset trả về PIL + id =====
# class ImageDataset(Dataset):
#     def __init__(self, image_dir):
#         self.image_dir = image_dir
#         self.image_files = [
#             f for f in os.listdir(image_dir)
#             if f.lower().endswith(('.jpg','.jpeg','.png')) and f not in ['6801.jpg','6803.jpg']
#         ]
#     def __len__(self): return len(self.image_files)
#     def __getitem__(self, idx):
#         fn = self.image_files[idx]
#         img = Image.open(os.path.join(self.image_dir, fn)).convert("RGB")
#         return img, os.path.splitext(fn)[0]

# def collate_pil(batch):
#     imgs, ids = zip(*batch)
#     return list(imgs), list(ids)

# # ===== Nạp OpenCLIP =====
# # ViT-B/32 nhẹ & phổ biến; có thể đổi 'ViT-L-14' + pretrained='laion2b_s32b_b82k' nếu GPU mạnh
# model_name = 'ViT-B-32'
# pretrained  = 'laion2b_s34b_b79k'

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained, device=device)
# model.eval()

# # ===== Hàm trích batch =====
# @torch.no_grad()
# def extract_batch_openclip(pil_images, device):
#     # preprocess trả sẵn tensor [B,3,224,224] (tuỳ backbone); stack thủ công
#     batch = torch.stack([preprocess(img) for img in pil_images], dim=0).to(device)
#     # AMP để tiết kiệm VRAM
#     use_amp = torch.cuda.is_available()
#     with torch.cuda.amp.autocast(enabled=use_amp):
#         feats = model.encode_image(batch)          # [B, D]
#     feats = feats / feats.norm(dim=-1, keepdim=True)
#     return feats.detach().cpu().numpy()            # (B, D)

# # ===== Loop DataLoader =====
# loader = DataLoader(ImageDataset(IMAGE_DIR),
#                     batch_size=16, shuffle=False, num_workers=2,
#                     pin_memory=True, collate_fn=collate_pil)

# features = {}
# for pil_batch, id_batch in tqdm(loader, desc="Extracting CLIP (open_clip)"):
#     feats = extract_batch_openclip(pil_batch, device)    # (B, D)
#     for k, v in zip(id_batch, feats):
#         features[k] = v[np.newaxis, :]                   # (1, D)

# with open(FEATURES_FILE, 'wb') as f:
#     pickle.dump(features, f)

# print("Saved:", FEATURES_FILE)